# Question 1: Can a student's other reasons predict whether the course was mandatory?

**The question:** Can we predict, from a student's *other* reasons for taking the course, whether they ticked **"This was mandatory for me"**?

## Step 1: Load the data

`read_csv` reads the file into a table (dataframe). `df.shape` gives **(rows, columns)**.

- We get 222 rows (students) and 15 columns (survey answers and engineered columns).

In [ ]:
import pandas as pd

df = pd.read_csv("../Data/Processed/Clean/dataminers_clean.csv")
df.shape

## Step 2: Filtering Rows

We keep only the rows we want to work with: **Software Design students from 2020-2025**.

- Students from other programmes have freely chosen the course as an elective. 
- In 2026 students can choose freely. Including it would hide the pattern.
- We now have 133 students.

In [ ]:
df = df[(df["programme_clean"] == "Software Design") & (df["response_year"] < 2026)]
df.shape

## Step 3: Feature Selection

- Label (`y`): the column we want to predict, `mandatory` (1 = mandatory, 0 = not).
- Features (`X`): the columns the model learns from, here the four motives: `exciting`, `commercial_skills`, `uncertain`, `less_attractive`.

`value_counts()` shows how balanced the label is: **78 not mandatory (0)** and **55 mandatory (1)**.

In [ ]:
features = ["exciting", "commercial_skills", "uncertain", "less_attractive"]

X = df[features]
y = df["mandatory"]

y.value_counts()

## Step 4: Splitting the Data

A model must be tested on students it has not seen. Otherwise we only measure how well it memorised the training data. So we split the data:

- 80% training data: the model learns from these students.
- 20% test data: we compare the model's guesses with the real answers.

Two settings:

- `random_state=42`: the same random split every time the code runs.
- `stratify=y`: the share of mandatory students is the same in both parts.

We get 106 training and 27 test students. 

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(X_train.shape, X_test.shape)

## Step 5: Baseline

A **baseline** is the simplest possible "model": always guess the most common class, here *not mandatory*.

Its accuracy is the share of test students who are not mandatory = **0.59**.

Our model has to beat this number. 

In [ ]:
print("Baseline accuracy:", (y_test == 0).mean())

## Step 6: Logistic regression

Logistic regression turns the features into a probability that the student is mandatory (using the sigmoid function), and predicts mandatory when the number exceeds the decision boundary at 0.5.

- `fit` lets the model learn from the training students only.
- `predict` makes it guess for the test students.
- `accuracy_score` is the share of correct guesses.

**Result: 0.67**

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))

## Step 7: Confusion matrix

The confusion matrix sorts the model's guesses into four boxes. 

The diagonal (12 and 6) is the model being right. The other two boxes are the false predictions.



In [ ]:
# Each box shows its name (TN, FP, FN, TP), what it means and the number of students.
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

cm = confusion_matrix(y_test, y_pred)

# the text written in each of the four boxes
text = [
    [f"TN\ncorrect: not mandatory\n{cm[0][0]} students", f"FP\nincorrect: called mandatory\n{cm[0][1]} students"],
    [f"FN\nincorrect: was mandatory\n{cm[1][0]} students", f"TP\ncorrect: mandatory\n{cm[1][1]} students"],
]

plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=text, fmt="", cmap="Blues", cbar=False,
            xticklabels=["Predicted\nnot mandatory", "Predicted\nmandatory"],
            yticklabels=["Actually\nnot mandatory", "Actually\nmandatory"])
plt.yticks(rotation=0)
plt.title("Logistic regression: confusion matrix")
plt.show()

### Metrics from the confusion matrix

All four metrics can be calculated from the four boxes. We use .accuracy_score() and .classification_report(), to get these metrics. 

| Metric | Formula | Question it answers |
|---|---|---|
| Accuracy | (TP + TN) / all | How many guesses were right? |
| Precision | TP / (TP + FP) | When the model says mandatory, how often is it right? |
| Recall | TP / (TP + FN) | Of the truly mandatory students, how many did it find? |
| F1 | 2 × (precision × recall) / (precision + recall) | One number that balances precision and recall |


In [ ]:
from sklearn.metrics import accuracy_score, classification_report
# Read the four boxes out of the confusion matrix (rows = actual, columns = predicted).
print("Logistic Regression accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))


- **Accuracy 0.67:** 18 of 27 test students were classified correctly. The baseline gets 0.59, so the model is only a tiny bit better.
- **Precision 0.60:** when the model says "mandatory", it is right 6 times out of 10.
- **Recall 0.55:** it finds 6 of the 11 truly mandatory students and misses 5.
- **F1 0.57:** a balance of the two. Both are middling, so F1 is too.

## Step 8: k-NN

We want to run the experiment with another model to see if we get different results. 
For each test student, k-NN finds the k most similar training students (closest in feature space) and takes their majority label. We use **k = 5**, as in the slides. **k** is a *hyperparameter*: we set it ourselves, the model does not learn it.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report

knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train, y_train)
y_pred_knn = knn.predict(X_test)

print("k-NN accuracy:", accuracy_score(y_test, y_pred_knn))
print(classification_report(y_test, y_pred_knn)) ## This is based on the same input as the confusion matrix. 

**Result:** accuracy **0.78** (21 of 27). For the mandatory class: precision **1.00**, recall **0.45**.

- When k-NN says "mandatory" it is always right (5 of 5), but it finds only 5 of the 11 truly mandatory students. 

k-NN scores higher on accuracy and F1, but that is 3 more students out of 27 on a single split. It is not strong evidence.

### Confusion matrix for k-NN

In [ ]:
# Confusion matrix for k-NN (k=5), in the same layout as the logistic regression one above
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

cm_knn = confusion_matrix(y_test, y_pred_knn)

# the text written in each of the four boxes
text_knn = [
    [f"TN\ncorrect: not mandatory\n{cm_knn[0][0]} students", f"FP\nincorrect: called mandatory\n{cm_knn[0][1]} students"],
    [f"FN\nincorrect: was mandatory\n{cm_knn[1][0]} students", f"TP\ncorrect: mandatory\n{cm_knn[1][1]} students"],
]

plt.figure(figsize=(7, 5))
sns.heatmap(cm_knn, annot=text_knn, fmt="", cmap="Blues", cbar=False,
            xticklabels=["Predicted\nnot mandatory", "Predicted\nmandatory"],
            yticklabels=["Actually\nnot mandatory", "Actually\nmandatory"])
plt.yticks(rotation=0)
plt.title("k-NN (k=5): confusion matrix")
plt.show()

### Choosing different k's

 A small k can **overfit** (it follows single odd points), a large k can **underfit** (it ignores the pattern). We compare accuracy on the training students with accuracy on the test students. Odd k values avoid tied votes.

**Result:** the accuracy barely changes (train 0.79-0.81, test 0.78 for every k).

There is nothing to tune, so we keep k = 5 for the report.

In [ ]:
for k in [1, 3, 5, 7, 9, 11, 15, 21]:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train, y_train)
    print(k, "train:", round(knn.score(X_train, y_train), 2),
             "test:", round(knn.score(X_test, y_test), 2))

## Conclusion

**Question:** Can a student's other reasons for taking the course predict whether they ticked "This was mandatory for me"?

**Answer:** Only a little. The motives carry some information, but not enough for a useful predictor.

| | Baseline | Logistic regression | k-NN (k=5) |
|---|---|---|---|
| Accuracy | 0.59 | 0.67 | 0.78 |
| Precision (mandatory) | - | 0.60 | 1.00 |
| Recall (mandatory) | - | 0.55 | 0.45 |
| F1 (mandatory) | - | 0.57 | 0.62 |

- Both models beat the baseline, but only by a few students out of 27, so we cannot say the difference is real.
- **Logistic regression**: finds slightly more of the mandatory students (recall 0.55) but calls some students mandatory by mistake (precision 0.60).
- **k-NN**: Whenever it says "mandatory" it is right (precision 1.00), but it misses more than half of the truly mandatory students (recall 0.45).
- Which model is "better" depends on which mistake is worse: missing a mandatory student (choose recall) or flagging a student wrongly (choose precision).
- The value of k hardly mattered.

## Why did the models do badly? (done afterwards)

This part was added after seeing the results, to find out why the models did so badly. We check how many of the mandatory students also ticked another motive.

In [ ]:
import matplotlib.pyplot as plt

mand_students = df[df["mandatory"] == 1]
other_motives = ["exciting", "commercial_skills", "uncertain", "less_attractive"]

# Number of mandatory students by how many other motives they ticked (0 to 4)
counts = mand_students[other_motives].sum(axis=1).value_counts().reindex(range(5), fill_value=0)

# Highlight the students who ticked nothing else
colors = ["tab:orange" if n == 0 else "steelblue" for n in counts.index]

plt.figure(figsize=(6, 4))
bars = plt.bar(counts.index, counts.values, color=colors)
plt.bar_label(bars)
plt.xlabel("Number of other motives ticked")
plt.ylabel("Mandatory students")
plt.title(f"Mandatory students by other motives ticked (n={len(mand_students)})")
plt.show()

The orange bar shows that 27 of the 55 mandatory students (about half) did not pick any other motive.

Now we compare with the other motives: of the students who ticked each motive, how many also ticked another one? Mandatory (51%) is lower than exciting, commercial_skills and less_attractive, and about the same as uncertain.

In [ ]:
# For each motive: of the students who ticked it, what share also ticked at least one other motive?
motives = ["exciting", "mandatory", "commercial_skills", "uncertain", "less_attractive"]
n_ticked = df[motives].sum(axis=1)          # how many motives each student ticked

share = {}
for m in motives:
    ticked_m = df[df[m] == 1]
    share[f"{m} (n={len(ticked_m)})"] = (n_ticked[ticked_m.index] >= 2).mean() * 100

share = pd.Series(share).sort_values()
colors = ["tab:orange" if name.startswith("mandatory") else "steelblue" for name in share.index]

plt.figure(figsize=(7, 4))
bars = plt.barh(share.index, share.values, color=colors)
plt.bar_label(bars, fmt="%.0f%%")
plt.xlabel("% who also ticked at least one other motive")
plt.xlim(0, 110)
plt.title("Students who ticked each motive and also ticked another")
plt.show()

The form requires at least one reason, so a student with no other motive must be mandatory. This applies to 27 of the 55 mandatory students - about half.

Mandatory students are also less likely than most other groups to tick a second motive: 51%, compared with 65% for exciting and 74% for commercial skills